# 01 — Auditoría de datos y evidencia EM2023

Este notebook reúne las auditorías de fuentes y de la base consolidada. Actualiza el inventario documental y los reportes agregados de calidad/relaciones; inspecciona la SQLite existente en solo lectura. No reconstruye las tablas, cambia fuentes ni escribe filas de microdatos en los reportes.

Entradas: `data/sources/raw/`, `docs/data_dictionary/source_schema.json` y `data/relational/em2023_relational.sqlite`. Salidas: inventario, esquema y auditorías agregadas bajo `docs/data_dictionary/`.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from mode_choice.source_inventory import write_catalogs
from mode_choice.quality_audit import audit_source_csvs, write_audit_reports
from mode_choice.source_readiness import inspect_source_readiness, write_readiness_report
from mode_choice.relational_validation import validate_relational_candidates, write_relational_reports

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), 'Run this notebook from the repository root.'

inventory_path, schema_path = write_catalogs(PROJECT_ROOT)
table_audit, column_audit = audit_source_csvs(PROJECT_ROOT)
quality_paths = write_audit_reports(PROJECT_ROOT, table_audit, column_audit)
readiness = inspect_source_readiness(PROJECT_ROOT)
readiness_path = write_readiness_report(PROJECT_ROOT, readiness)
candidate_results, relation_results = validate_relational_candidates(PROJECT_ROOT)
relationship_paths = write_relational_reports(PROJECT_ROOT, candidate_results, relation_results)

database_path = PROJECT_ROOT / 'data/relational/em2023_relational.sqlite'
if not database_path.is_file():
    raise FileNotFoundError('The existing relational SQLite database is required; this audit does not rebuild it.')
connection = sqlite3.connect(f'file:{database_path.as_posix()}?mode=ro', uri=True)
try:
    integrity = connection.execute('PRAGMA integrity_check').fetchone()[0]
    foreign_key_issues = connection.execute('PRAGMA foreign_key_check').fetchall()
    table_names = [row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    table_counts = {name: connection.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0] for name in table_names}
finally:
    connection.close()

if integrity != 'ok' or foreign_key_issues:
    raise RuntimeError(f'SQLite integrity check failed: {integrity}; foreign-key issues={len(foreign_key_issues)}')

print({'inventory': inventory_path.relative_to(PROJECT_ROOT).as_posix(),
       'schema': schema_path.relative_to(PROJECT_ROOT).as_posix(),
       'quality_reports': [p.relative_to(PROJECT_ROOT).as_posix() for p in quality_paths],
       'readiness_report': readiness_path.relative_to(PROJECT_ROOT).as_posix(),
       'relationship_reports': [p.relative_to(PROJECT_ROOT).as_posix() for p in relationship_paths],
       'sqlite_integrity': integrity, 'sqlite_tables': table_counts})

## Evidencia reciente de Etapa IV

P4 documenta el control MIC y la georreferenciación; P5 describe la estructura y los conteos de la base final; P6 documenta la estimación por factor del hogar y la fórmula de viajes. Los documentos son evidencia metodológica secundaria del contrato: las reglas de variables y valores fuente se preservan y las diferencias se registran, no se corrigen por inferencia.